In [5]:
import pandas as pd
import numpy as np
import gymnasium as gym
from gymnasium import spaces
import warnings
warnings.filterwarnings('ignore')

# =============================================
# 1. LOAD & PREPARE DATA
# =============================================
state_data = pd.read_csv("../data/state_dataset.csv", index_col="Date", parse_dates=True)
market_data = pd.read_csv("../data/market_master.csv", index_col="Date", parse_dates=True)

# Merge only what's needed
data = state_data.join(market_data[["nifty_ret"]], how="inner")
print(f"Dataset shape: {data.shape}")
print("Columns:", data.columns.tolist())

# =============================================
# 2. REWARD FUNCTION (Correct & Clean)
# =============================================
class RewardFunction:
    def __init__(self, cost=0.0005, risk_penalty=0.08, drawdown_penalty=0.25):
        self.cost = cost
        self.risk_penalty = risk_penalty
        self.drawdown_penalty = drawdown_penalty
    
    def compute(self, current_ret, prev_position, new_position, drawdown):
        # Base strategy return from PREVIOUS position
        strategy_ret = prev_position * current_ret
        
        # Transaction cost only when changing position
        cost = abs(new_position - prev_position) * self.cost
        
        # Risk penalty (quadratic on return volatility)
        risk_penalty = self.risk_penalty * (strategy_ret ** 2)
        
        # Drawdown penalty
        dd_penalty = self.drawdown_penalty * abs(drawdown) if drawdown < -0.05 else 0
        
        # Final reward
        reward = strategy_ret - cost - risk_penalty - dd_penalty
        
        # Net return for equity update
        net_ret = strategy_ret - cost
        
        return reward, net_ret


# =============================================
# 3. QUANTUM ALPHA RL ENVIRONMENT
# =============================================
class QuantumAlphaEnv(gym.Env):
    
    def __init__(self, data, initial_balance=1_000_000):
        super().__init__()
        
        self.data = data.reset_index(drop=True)
        self.max_steps = len(self.data) - 1
        self.initial_balance = initial_balance
        
        # Action space: 0=Flat, 1=Long, 2=Short
        self.action_space = spaces.Discrete(3)
        
        # Observation space: only normalized features (no raw returns)
        self.feature_cols = [
            "mom_20_norm", "vol_signal_norm", "trend_signal_norm",
            "dd_signal_norm", "vix_signal_norm", "breadth_signal_norm"
        ]
        
        self.observation_space = spaces.Box(
            low=-5.0, high=5.0,
            shape=(len(self.feature_cols),),
            dtype=np.float32
        )
        
        self.reward_fn = RewardFunction()
        self.reset()
    
    def reset(self):
        self.current_step = 0
        self.position = 0
        self.balance = self.initial_balance
        self.peak_balance = self.initial_balance
        return self._get_observation()
    
    def _get_observation(self):
        row = self.data.iloc[self.current_step]
        obs = row[self.feature_cols].values.astype(np.float32)
        return obs
    
    def step(self, action):
        # Convert action to position
        new_position = {0: 0, 1: 1, 2: -1}[action]
        prev_position = self.position
        
        # Market return at current step
        current_ret = self.data.iloc[self.current_step]["nifty_ret"]
        
        # Current drawdown
        current_drawdown = (self.balance / self.peak_balance) - 1 if self.peak_balance > 0 else 0
        
        # Compute reward using advanced reward function
        reward, net_ret = self.reward_fn.compute(
            current_ret=current_ret,
            prev_position=prev_position,
            new_position=new_position,
            drawdown=current_drawdown
        )
        
        # Update balance using NET return
        self.balance *= (1 + net_ret)
        
        # Update peak balance
        self.peak_balance = max(self.peak_balance, self.balance)
        
        # Update position
        self.position = new_position
        
        # Move to next step
        self.current_step += 1
        done = self.current_step >= self.max_steps
        
        return self._get_observation(), reward, done, {
            "balance": self.balance,
            "position": self.position,
            "drawdown": (self.balance / self.peak_balance) - 1
        }
    
    def render(self):
        print(f"Step: {self.current_step} | Position: {self.position} | Balance: {self.balance:,.2f}")

# =============================================
# 4. CREATE ENVIRONMENT + QUICK TEST
# =============================================
env = QuantumAlphaEnv(data)

print("✅ Quantum Alpha RL Environment Created Successfully!")
print(f"Observation shape: {env.observation_space.shape}")
print(f"Action space: Discrete(3)")
print(f"Total steps: {env.max_steps}")

# Quick random agent test
print("\n--- Random Agent Test ---")
obs = env.reset()
total_reward = 0

for step in range(100):
    action = env.action_space.sample()
    obs, reward, done, info = env.step(action)
    total_reward += reward
    if step % 20 == 0:
        print(f"Step {step}: Reward={reward:.5f}, Balance={info['balance']:,.2f}")
    if done:
        break

print(f"\nTest Completed")
print(f"Total Reward: {total_reward:.5f}")
print(f"Final Balance: {info['balance']:,.2f}")

Dataset shape: (1529, 7)
Columns: ['mom_20_norm', 'vol_signal_norm', 'trend_signal_norm', 'dd_signal_norm', 'vix_signal_norm', 'breadth_signal_norm', 'nifty_ret']
✅ Quantum Alpha RL Environment Created Successfully!
Observation shape: (6,)
Action space: Discrete(3)
Total steps: 1528

--- Random Agent Test ---
Step 0: Reward=-0.00050, Balance=999,500.00
Step 20: Reward=0.00670, Balance=1,032,253.82
Step 40: Reward=-0.03918, Balance=1,079,680.58
Step 60: Reward=-0.06568, Balance=861,354.80
Step 80: Reward=-0.06149, Balance=850,868.89

Test Completed
Total Reward: -3.36772
Final Balance: 852,277.53
